# F1 Pit Stop Prediction - Baseline Model
**Kaggle Playground Series S6E5**

Simple LightGBM with raw features and GroupKFold CV. Goal is to get a clean first submission on the board.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
sub   = pd.read_csv('../data/sample_submission.csv')

print('Train:', train.shape)
print('Test: ', test.shape)

## Minimal preprocessing

In [ ]:
# Encode categoricals with simple label encoding for baseline
cat_cols = ['Driver', 'Compound', 'Race']

for col in cat_cols:
    mapping = {v: i for i, v in enumerate(train[col].unique())}
    train[col + '_enc'] = train[col].map(mapping)
    test[col + '_enc']  = test[col].map(mapping).fillna(-1).astype(int)

drop_cols = ['id', 'PitNextLap', 'Driver', 'Compound', 'Race']
features  = [c for c in train.columns if c not in drop_cols]

X      = train[features]
y      = train['PitNextLap']
X_test = test[features]

print('Features used:', len(features))
print(features)

## GroupKFold cross-validation

In [ ]:
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': 4,
    'verbose': -1,
    'random_state': 42
}

gkf    = GroupKFold(n_splits=5)
groups = train['Race'] + '_' + train['Year'].astype(str)

oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_aucs  = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'Fold {fold+1}/5 ...')
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)]
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds        += model.predict_proba(X_test)[:, 1] / 5

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold+1} AUC: {fold_auc:.5f}')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOOF AUC: {oof_auc:.5f}')
print(f'Fold AUCs: {[round(a, 5) for a in fold_aucs]}')

## Feature importance

In [ ]:
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind='barh', x='feature', y='importance', ax=ax, legend=False)
ax.set_title('Feature importance (last fold)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Save submission

In [ ]:
sub['PitNextLap'] = test_preds
sub.to_csv('../submissions/submission_lgb_baseline.csv', index=False)
print('Saved submission_lgb_baseline.csv')
print(f'OOF AUC: {oof_auc:.5f}')